### Threshold selection of numerical features of dispersion

In [6]:
from sklearn import datasets
from sklearn.feature_selection import VarianceThreshold

# Load the iris dataset
iris = datasets.load_iris()

# Create the feature set and target set
features = iris.data
target = iris.target

# Create a VarianceThreshold object with a threshold of 0.5
thresholder = VarianceThreshold(threshold=0.5)

# Fit the VarianceThreshold object to the features and transform the features
features_high_variance = thresholder.fit_transform(features)

# Print the first three rows of the features with high variance
features_high_variance[0:3]


array([[5.1, 1.4, 0.2],
       [4.9, 1.4, 0.2],
       [4.7, 1.3, 0.2]])

In [9]:
thresholder.fit(features).variances_


array([0.68112222, 0.18871289, 3.09550267, 0.57713289])

### Threshold selection of binary feature variance

In [11]:
from sklearn.feature_selection import VarianceThreshold

# Create a VarianceThreshold object with a threshold of 0.2
thresholder = VarianceThreshold(threshold=0.2)

# Feature 0: 80% class 0
# Feature 1: 80% class 1
# Feature 2: 60% class 0, 40% class 1
features = [
    [0, 1, 0],
    [0, 1, 1],
    [0, 1, 0],
    [0, 1, 1],
    [1, 0, 0]
]

# Fit the VarianceThreshold object to the features and transform the features
thresholder = VarianceThreshold(threshold=(.75 * (1 - .75)))
features_high_variance = thresholder.fit_transform(features)
features_high_variance

array([[0],
       [1],
       [0],
       [1],
       [0]])

### Processing features with a high correlation coefficient

In [21]:
import numpy as np
import pandas as pd

# Create a DataFrame with the features
features = np.array(
    [
        [1, 1, 1],
        [2, 2, 0],
        [3, 3, 1],
        [4, 4, 0],
        [5, 5, 1],
        [6, 6, 0],
        [7, 7, 1],
        [8, 7, 0],
        [9, 7, 1],
    ]
)

# Create a DataFrame with the features
dataframe = pd.DataFrame(features)

# Create a correlation matrix
corr_matrix = dataframe.corr().abs()

# Select the upper triangle of the correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features with correlation greater than 0.95
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]

# Drop features with correlation greater than 0.95
dataframe.drop(dataframe.columns[to_drop], axis=1).head(10)

,0,2
0,1,1
1,2,0
2,3,1
3,4,0
4,5,1
5,6,0
6,7,1
7,8,0
8,9,1


In [22]:
# Create a correlation matrix
dataframe.corr()


,0,1,2
0,1.000000,0.976103,0.000000
1,0.976103,1.000000,-0.034503
2,0.000000,-0.034503,1.000000


In [23]:
upper

,0,1,2
0,NaN,0.976103,0.000000
1,NaN,NaN,0.034503
2,NaN,NaN,NaN


### Removing irrelevant features for classification

In [27]:
from sklearn.datasets import load_iris
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2, f_classif

# Load the iris dataset
iris = load_iris()
features = iris.data
target = iris.target

# Transform the features to integers for chi-squared test
features = features.astype(int)

# Select two features with the highest chi-squared scores
chi2_selector = SelectKBest(chi2, k=2)
features_kbest = chi2_selector.fit_transform(features, target)

# Print the original and reduced number of features
print("Original number of features:", features.shape[1])
print("Reduced number of features:", features_kbest.shape[1])

Original number of features: 4
Reduced number of features: 2


In [28]:
# Select two features with the highest F-values
fvalue_selector = SelectKBest(f_classif, k=2)
features_kbest = fvalue_selector.fit_transform(features, target)

# Print the original and reduced number of features
print("Original number of features:", features.shape[1])
print("Reduced number of features:", features_kbest.shape[1])


Original number of features: 4
Reduced number of features: 2


In [29]:
from sklearn.feature_selection import SelectPercentile

# Select 75% of features with the highest F-values
fvalue_selector = SelectPercentile(f_classif, percentile=75)
features_kbest = fvalue_selector.fit_transform(features, target)

# Print the original and reduced number of features
print("Original number of features:", features.shape[1])
print("Reduced number of features:", features_kbest.shape[1])

Original number of features: 4
Reduced number of features: 3


### Recursive feature elimination

In [33]:
import warnings
from sklearn.datasets import make_regression
from sklearn.feature_selection import RFECV
from sklearn import datasets, linear_model

# Disable warnings related to the internal gelsd function in scipy
warnings.filterwarnings(action="ignore", module="scipy",
                        message="^internal gelsd")

# Создамь матрицу признаков, целевой вектор и настоящие коэффициенты
features, target = make_regression(
    n_samples=10000,
    n_features=100,
    n_informative=2,
    random_state=1)

# Create a linear regression model
ols = linear_model.LinearRegression()

# Rcursive Feature Elimination with Cross-Validation (RFECV) to select features
rfecv = RFECV(estimator=ols, step=1, scoring="neg_mean_squared_error")
rfecv.fit(features, target)
rfecv.transform(features)





array([[ 0.00850799,  0.7031277 ],
       [-1.07500204,  2.56148527],
       [ 1.37940721, -1.77039484],
       ...,
       [-0.80331656, -1.60648007],
       [ 0.39508844, -1.34564911],
       [-0.55383035,  0.82880112]], shape=(10000, 2))

In [34]:
# Print the number of features selected by RFECV
rfecv.n_features_


np.int64(2)

In [35]:
# Print the support mask of selected features by RFECV
rfecv.support_


array([False, False, False, False, False,  True, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False,  True, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False])

In [36]:
# Print the ranking of features by RFECV
rfecv.ranking_

array([ 7, 55, 25, 68, 57,  1, 89, 96,  4, 85, 58, 82, 78, 23, 16, 88, 87,
       24, 29, 73, 95, 76, 64, 97, 94, 86, 93, 38, 11, 59, 52, 79, 35, 71,
       22,  5, 99, 81, 75,  1, 46, 61, 62, 77, 42, 72, 70, 54, 53, 56, 31,
       90,  9, 98, 74, 45, 84, 80, 27, 43, 15, 17, 91, 51,  3, 83, 19, 10,
       66, 50, 34, 67, 40, 36, 41, 69, 33, 21, 49, 26, 48, 63, 30, 12, 92,
       13, 37,  8, 65, 39,  6, 44, 14,  2, 18, 32, 60, 20, 47, 28])